### Zircon U-Pb Table ###
This notebook contains code used to create the supplementary table containing all original zircon U-Pb data.

In [11]:
# Imports
import os
import pandas as pd
import numpy as np

In [12]:
# Identify directory for U-Pb data
directory = 'laserchron_data/'

In [13]:
# Make a class for raw  data
class RawData:
    def __init__(self,name,accepted,rejected,standards,syst_238,syst_207):
        self.name = name
        self.accepted = accepted
        self.rejected = rejected
        self.standards = standards
        self.syst_238 = syst_238
        self.syst_207 = syst_207
        return

In [14]:
samples = []
skiprows=26

In [15]:
# Get individual igneous datatables and clean

for root,dirs,files in os.walk(directory):
    files.sort()
    for file in files:
        if ('VanTongeren' in file) & file.endswith('.xlsx'):
            path = os.path.join(root,file)

            # Read relevant part of the Excel file
            df = pd.read_excel(path,skiprows=skiprows,usecols='A:T',header=None,sheet_name='Sheet1')
            df.columns = np.arange(20)

            reject = pd.read_excel(path,skiprows=skiprows,usecols='W:AP',header=None,sheet_name='Sheet1')
            reject.columns = np.arange(20)

            stand = pd.read_excel(path,skiprows=skiprows,usecols='AS:BL',header=None,sheet_name='Sheet1')
            stand.columns = np.arange(20)

            syst = pd.read_excel(path,skiprows=9,nrows=2,header=None,
                                        usecols='B')

            syst_238 = syst.loc[0,1]
            syst_207 = syst.loc[1,1]
            
            # Remove empty rows
            df.dropna(how='all',inplace=True)
            reject.dropna(how='all',inplace=True)
            
            # Move SLs to standards
            sl_bool = df.iloc[:,0].str.contains('SL')

            sl = df[sl_bool==True]
            df = df[sl_bool==False]

            # Remove rejected standards
            reject.iloc[:,0] = reject.iloc[:,0].astype(str)

            std_reject = (
                (reject.iloc[:,0].str.contains('SL'))|
                (reject.iloc[:,0].str.contains('F'))|
                (reject.iloc[:,0].str.contains('R33'))
            )

            reject = reject[std_reject==False]


            # Insert new columns for 238-235 concordance
            age238 = df.loc[:,11]
            age238_rej = reject.loc[:,11]

            age235 = df.loc[:,13]
            age235_rej = reject.loc[:,13]

            conc238235 = (age238/age235)*100
            conc238235_rej = (age238_rej/age235_rej)*100

            df.loc[:,20] = conc238235
            reject.loc[:,20] = conc238235_rej

            # Get names from filename
            file_parts = file.split(' ')
            names = [part for part in file_parts if part.startswith('MR24')]
            
            for name in names:

                df_ind = df[df.iloc[:,0].str.contains(name)]
                reject_ind = reject[reject.iloc[:,0].str.contains(name)]

                sample = RawData(name=name,accepted=df_ind,rejected=reject_ind,standards=stand,syst_238=syst_238,syst_207=syst_207)
                print(sample.name)

                samples.append(sample)

MR24-10
MR24-12
MR24-13
MR24-15
MR24-16


In [16]:
# Get individual DZ data tables and clean

for root,dirs,files in os.walk(directory):
    files.sort()
    for file in files:
        if ('MR24-0' in file) & file.endswith('.xlsx'):
            path = os.path.join(root,file)
            
            # Read relevant part of the Excel file
            df = pd.read_excel(path,skiprows=skiprows,usecols='A:T',header=None,sheet_name='Sheet1')
            df.columns = np.arange(20)

            reject = pd.read_excel(path,skiprows=skiprows,usecols='W:AP',header=None,sheet_name='Sheet1')
            reject.columns = np.arange(20)

            stand = pd.read_excel(path,skiprows=skiprows,usecols='AS:BL',header=None,sheet_name='Sheet1')
            stand.columns = np.arange(20)

            syst = pd.read_excel(path,skiprows=9,nrows=2,header=None,
                                        usecols='B')

            syst_238 = syst.loc[0,1]
            syst_207 = syst.loc[1,1]
            
            # Remove empty rows
            df.dropna(how='all',inplace=True)
            reject.dropna(how='all',inplace=True)
            
            # Move SLs to standards
            sl_bool = df.iloc[:,0].str.contains('SL')

            sl = df[sl_bool==True]
            df = df[sl_bool==False]

            # Remove rejected standards
            reject.iloc[:,0] = reject.iloc[:,0].astype(str)

            std_reject = (
                (reject.iloc[:,0].str.contains('SL'))|
                (reject.iloc[:,0].str.contains('F'))|
                (reject.iloc[:,0].str.contains('R33'))
            )

            reject = reject[std_reject==False]


            # Insert new columns for 238-235 concordance
            age238 = df.loc[:,11]
            age238_rej = reject.loc[:,11]

            age235 = df.loc[:,13]
            age235_rej = reject.loc[:,13]

            conc238235 = (age238/age235)*100
            conc238235_rej = (age238_rej/age235_rej)*100

            df.loc[:,20] = conc238235
            reject.loc[:,20] = conc238235_rej
            
            # Get name from file name
            name = file.split(' ')[1]

            sample = RawData(name=name,accepted=df,rejected=reject,standards=stand,syst_238=syst_238,syst_207=syst_207)
            print(sample.name)

            samples.append(sample)


MR24-006
MR24-008
MR24-009
MR24-017
MR24-020
MR24-023


In [17]:
# Set up the table

title = 'Table S1: Results of U-Th-Pb analyses for igneous and detrital zircon samples'
ncols = 21

title_row = pd.Series(index=np.arange(ncols),dtype='str')
title_row[0] = title
title_row[1:] = ''

header1 = pd.Series(index=np.arange(ncols),dtype='str')
header1[6] = 'Isotope ratios'
header1[11] = 'Apparent ages (Ma)'
header1[19] = 'Concordance'
header1[:6] = ''
header1[12:18] = ''
header1[20:] = ''

header2_vals = (
['Analysis','U','206Pb','U/Th','206Pb*','±2s','207Pb*','±2s','206Pb*','±2s','error','206Pb*','±2s','207Pb*','±2s','206Pb*','±2s','Best age','±2s','68 vs. 67', '68 vs. 75']
)
header2 = pd.Series(header2_vals)

header3_vals = (
    ['','(ppm)','204Pb','','207Pb*','(%)','235U*','(%)','238U','(%)','corr.','238U*','(Ma)','235U','(Ma)','207Pb*','(Ma)','(Ma)','(Ma)','(%)','(%)']
)

header3 = pd.Series(header3_vals)

empty_series = pd.Series(index=np.arange(ncols),dtype='str')
empty_series[:] = ''

reject_row = pd.Series(index=np.arange(ncols),dtype='str')
reject_row[0] = 'Rejected Analyses'

In [21]:
# Create large Pandas dataframe

df = pd.DataFrame(columns = np.arange(ncols))
df.loc[0,:] = title_row
df.loc[1,:] = empty_series
df.loc[2,:] = header1
df.loc[3,:] = empty_series
df.loc[4,:] = header2
df.loc[5,:] = header3
df.loc[6,:] = empty_series

new_names = ['JBD1','JBD2','JBD3','JBD4','JBD5','RH1','JB2','JB1','JB3','MHA1','MHA2']

for k, sample in enumerate(samples):
    sample.nickname = new_names[k]
    
    nick_row = pd.Series(index=np.arange(ncols),dtype='str')
    nick_row[0] = sample.nickname + ' (' + sample.name + ')'
    nick_row[1:] = ''

    df.loc[df.shape[0],:] = nick_row

    df = pd.concat([df,sample.accepted],axis=0,ignore_index=True)

    df.loc[df.shape[0],:] = empty_series

    if len(sample.rejected)>0:
        df.loc[df.shape[0],:] = reject_row
        df = pd.concat([df,sample.rejected],axis=0,ignore_index=True)
        df.loc[df.shape[0],:] = empty_series

print(df)

# Save the dataframe to a CSV
df.to_csv('table.csv',encoding='utf-8-sig',header=False,index=False)

                                                     0      1         2   \
0     Table S1: Results of U-Th-Pb analyses for igne...                    
1                                                                          
2                                                                          
3                                                                          
4                                              Analysis      U     206Pb   
...                                                 ...    ...       ...   
2129                                       MR24-023 266  354.0  193627.0   
2130                                       MR24-023 272  256.0  916542.0   
2131                                       MR24-023 285  191.0  268325.0   
2132                                       MR24-023 296   45.0  111368.0   
2133                                                                       

         3        4    5               6    7         8    9   ...  \
0                